In [1]:
import pandas as pd  # Import pandas for structured data analysis.
import numpy as np  # Import NumPy for numerical diagnostics.
from pathlib import Path  # Import Path for file-safe references.
from IPython.display import display  # Import display for readable audit tables.
from scipy import stats  # Import statistical tests for audit checks.
FILE_PATH = "raw_data/support_tickets.csv"  # Point to the project raw dataset.
audit_findings = []  # Create a register for evidence-based findings.
print("BUSINESS INSIGHT: Customer Retention and Churn")  # State the consulting context for the audit.
print("BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.")  # State the business problem being investigated.
print("AUDIT LENS: retention, repeat purchase, payment friction, service experience")  # State the signals relevant to this problem.


BUSINESS INSIGHT: Customer Retention and Churn
BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.
AUDIT LENS: retention, repeat purchase, payment friction, service experience


In [2]:
path_candidates = [
    Path(FILE_PATH),
    Path.cwd() / FILE_PATH,
    Path.cwd().parent / FILE_PATH,
    Path.cwd().parent.parent / FILE_PATH,
    Path.cwd().parent.parent.parent / FILE_PATH,
]

data_path = next((path for path in path_candidates if path.is_file()), None)

if data_path is None:
    raise FileNotFoundError(
        f"Could not locate {FILE_PATH!r}. Checked: "
        + ", ".join(str(path) for path in path_candidates)
    )

df = pd.read_csv(data_path)

print(f"Loaded {data_path.name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())  # Load the raw dataset before making changes.
print(f"Loaded {Path(FILE_PATH).name}")  # Confirm the source dataset loaded.
print(f"Rows: {len(df):,}")  # Show the available observation count.
print(f"Columns: {len(df.columns):,}")  # Show the available field count.
display(df.head())  # Inspect representative raw records.

Loaded support_tickets.csv
Rows: 9,000
Columns: 10


,ticket_id,customer_id,opened_at,closed_at,issue_type,channel,priority,agent_id,resolution_status,satisfaction_score
0,TKT-0000001,CUS-009096,2024-01-31 21:00:00,2024-02-01 03:00:00,Returns,Chat,Critical,AGT-315,Resolved,5.0
1,TKT-0000002,CUS-002221,2025-12-17 16:00:00,2025-12-20 05:00:00,Payment,Chat,Low,AGT-252,Resolved,NaN
2,TKT-0000003,CUS-011324,2023-12-04 18:00:00,2023-12-08 16:00:00,Returns,Email,Low,AGT-092,Resolved,NaN
3,TKT-0000004,CUS-002213,2024-05-02 23:00:00,2024-05-03 06:00:00,Delivery,Email,Medium,AGT-150,Resolved,NaN
4,TKT-0000005,CUS-007333,2021-10-15 08:00:00,2021-10-18 20:00:00,Returns,Store,Medium,AGT-020,Resolved,NaN


Loaded support_tickets.csv
Rows: 9,000
Columns: 10


,ticket_id,customer_id,opened_at,closed_at,issue_type,channel,priority,agent_id,resolution_status,satisfaction_score
0,TKT-0000001,CUS-009096,2024-01-31 21:00:00,2024-02-01 03:00:00,Returns,Chat,Critical,AGT-315,Resolved,5.0
1,TKT-0000002,CUS-002221,2025-12-17 16:00:00,2025-12-20 05:00:00,Payment,Chat,Low,AGT-252,Resolved,NaN
2,TKT-0000003,CUS-011324,2023-12-04 18:00:00,2023-12-08 16:00:00,Returns,Email,Low,AGT-092,Resolved,NaN
3,TKT-0000004,CUS-002213,2024-05-02 23:00:00,2024-05-03 06:00:00,Delivery,Email,Medium,AGT-150,Resolved,NaN
4,TKT-0000005,CUS-007333,2021-10-15 08:00:00,2021-10-18 20:00:00,Returns,Store,Medium,AGT-020,Resolved,NaN


In [3]:
overview = pd.DataFrame({"metric":["rows","columns","duplicates","missing_cells"],"value":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum())]})  # Build an initial data-quality summary.
display(overview)  # Review the initial quality position.
print("Decision point: determine which findings require remediation.")  # Make the audit decision explicit.


,metric,value
0,rows,9000
1,columns,10
2,duplicates,0
3,missing_cells,5144


Decision point: determine which findings require remediation.


In [4]:
schema = pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"non_null":df.notna().sum().values,"missing":df.isna().sum().values,"unique":df.nunique(dropna=True).values})  # Profile schema completeness and cardinality.
display(schema)  # Inspect field-level structural evidence.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()  # Identify numeric fields for statistical checks.
text_cols = df.select_dtypes(include=["object","string"]).columns.tolist()  # Identify text fields for categorical checks.
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]  # Identify likely temporal fields.


,column,dtype,non_null,missing,unique
0,ticket_id,str,9000,0,9000
1,customer_id,str,9000,0,6319
2,opened_at,str,9000,0,8339
3,closed_at,str,8120,880,7585
4,issue_type,str,9000,0,21
5,channel,str,9000,0,5
6,priority,str,9000,0,4
7,agent_id,str,9000,0,350
8,resolution_status,str,9000,0,3
9,satisfaction_score,float64,4736,4264,5


In [5]:
missing = df.isna().sum().sort_values(ascending=False)  # Measure explicit missingness by field.
missing = missing[missing.gt(0)]  # Keep only fields with missing values.
display(missing.to_frame("missing_count"))  # Inspect missing-value concentration.
for col in missing.index: audit_findings.append({"issue":"missing_values","column":col,"count":int(missing[col])})  # Register observed missing-value evidence.


,missing_count
satisfaction_score,4264
closed_at,880


In [6]:
duplicates = int(df.duplicated().sum())  # Measure exact duplicate records.
audit_findings.append({"issue":"duplicate_rows","count":duplicates})  # Register duplicate-row evidence.
print(f"Duplicate rows identified: {duplicates:,}")  # Report duplicate-row evidence.


Duplicate rows identified: 0


In [7]:
numeric_audit = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()  # Summarise numeric distributions.
display(numeric_audit)  # Inspect scale, spread and potential extremes.
if numeric_cols: outlier_rates = ((df[numeric_cols] < df[numeric_cols].quantile(.25) - 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25))) | (df[numeric_cols] > df[numeric_cols].quantile(.75) + 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25)))).mean().sort_values(ascending=False)  # Estimate IQR-based extreme-value rates.
if numeric_cols: display(outlier_rates.to_frame("iqr_extreme_rate"))  # Inspect fields requiring business review.


,count,mean,std,min,25%,50%,75%,max
satisfaction_score,4736.0,3.844806,1.176228,1.0,3.0,4.0,5.0,5.0


,iqr_extreme_rate
satisfaction_score,0.0


In [8]:
category_audit = []  # Create categorical consistency checks.
for col in text_cols: category_audit.append({"column":col,"unique":int(df[col].nunique(dropna=True)),"blank":int(df[col].astype("string").str.strip().eq("").sum()),"top_values":df[col].value_counts(dropna=False).head(5).to_dict()})  # Profile text fields for inconsistent values.
display(pd.DataFrame(category_audit))  # Review categorical concentration and blanks.


,column,unique,blank,top_values
0,ticket_id,9000,0,"{'TKT-0000001': 1, 'TKT-0000002': 1, 'TKT-0000..."
1,customer_id,6319,0,"{'CUS-010011': 6, 'CUS-001434': 6, 'CUS-000859..."
2,opened_at,8339,0,"{'2023-07-09 12:00:00': 4, '2024-11-19 03:00:0..."
3,closed_at,7585,0,"{nan: 880, '2020-04-30 00:00:00': 4, '2023-07-..."
4,issue_type,21,0,"{'Delivery': 2542, 'Returns': 1770, 'Product Q..."
5,channel,5,0,"{'Email': 1822, 'Social': 1809, 'Store': 1807,..."
6,priority,4,0,"{'Medium': 3998, 'Low': 3227, 'High': 1503, 'C..."
7,agent_id,350,0,"{'AGT-299': 45, 'AGT-272': 43, 'AGT-062': 39, ..."
8,resolution_status,3,0,"{'Resolved': 6757, 'Closed': 1363, 'Open': 880}"


In [9]:
date_audit = []  # Create temporal field diagnostics.
for col in date_cols: parsed = pd.to_datetime(df[col], errors="coerce"); date_audit.append({"column":col,"parse_failures":int(parsed.isna().sum()-df[col].isna().sum()),"min":parsed.min(),"max":parsed.max()})  # Test temporal fields for parseability and range.
display(pd.DataFrame(date_audit))  # Review date integrity before analysis.


""


In [10]:
identifier_audit = []  # Create identifier uniqueness diagnostics.
for col in df.columns: identifier_audit.append({"column":col,"unique_ratio":round(df[col].nunique(dropna=True)/max(len(df),1),3)})  # Measure field-level uniqueness.
identifier_audit = pd.DataFrame(identifier_audit).sort_values("unique_ratio",ascending=False)  # Rank potential identifiers and keys.
display(identifier_audit.head(15))  # Inspect candidate identifiers and high-cardinality fields.


,column,unique_ratio
0,ticket_id,1.000
2,opened_at,0.927
3,closed_at,0.843
1,customer_id,0.702
7,agent_id,0.039
4,issue_type,0.002
5,channel,0.001
9,satisfaction_score,0.001
6,priority,0.000
8,resolution_status,0.000


In [11]:
numeric_pairs = []  # Create relationship diagnostics for numeric fields.
if len(numeric_cols) > 1: numeric_pairs = df[numeric_cols].corr(numeric_only=True).stack().reset_index(name="correlation")  # Measure numeric relationships for diagnostic context.
if numeric_pairs != []: display(numeric_pairs.sort_values("correlation",key=lambda s:s.abs(),ascending=False).head(20))  # Inspect strongest observed numeric relationships.


In [12]:
finding_table = pd.DataFrame(audit_findings)  # Convert findings into a reviewable audit register.
if finding_table.empty: finding_table = pd.DataFrame([{ "issue":"none_detected_by_template", "count":0 }])  # Record when automated checks find no issues.
display(finding_table)  # Review the evidence before remediation.
print("Consulting decision: validate material findings against business rules before cleaning.")  # Prevent automatic treatment of every anomaly as an error.


,issue,column,count
0,missing_values,satisfaction_score,4264
1,missing_values,closed_at,880
2,duplicate_rows,NaN,0


Consulting decision: validate material findings against business rules before cleaning.


In [13]:
stem = Path(FILE_PATH).stem  # Capture the dataset stem for output naming.
audit_summary = pd.DataFrame({"dataset":[Path(FILE_PATH).name],"rows":[len(df)],"columns":[len(df.columns)],"duplicates":[duplicates],"missing_cells":[int(df.isna().sum().sum())]})  # Create an auditable executive summary.
display(audit_summary)  # Present the final audit snapshot.
audit_summary.to_csv(f"outputs/{stem}_audit_summary.csv",index=False)  # Save the audit summary for downstream review.


,dataset,rows,columns,duplicates,missing_cells
0,support_tickets.csv,9000,10,0,5144
